# Python: Loss Development Triangle and Reserving

## Import Packages
### chainladder package: for creating loss triangle

In [3]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import chainladder as cl

## Data Processing
### 1. Read in data

In [5]:
url = "https://www.casact.org/sites/default/files/2026-03/ppauto_pos98-07%20%281%29.csv"
data = pd.read_csv(url)

In [6]:
data.head()

,GRCODE,GRNAME,AccidentYear,DevelopmentYear,DevelopmentLag,IncurredLosses,CumPaidLoss,BulkLoss,EarnedPremDIR,EarnedPremCeded,EarnedPremNet,Single,PostedReserves2007
0,43,IDS Property Cas Ins Co,1998,1998,1,50320,12762,21720,63709,3071,60638,0,277117.746
1,43,IDS Property Cas Ins Co,1998,1999,2,41036,26291,6514,63709,3071,60638,0,277117.746
2,43,IDS Property Cas Ins Co,1998,2000,3,39688,32420,2858,63709,3071,60638,0,277117.746
3,43,IDS Property Cas Ins Co,1998,2001,4,39278,36282,757,63709,3071,60638,0,277117.746
4,43,IDS Property Cas Ins Co,1998,2002,5,40647,38260,1444,63709,3071,60638,0,277117.746


### 2. Store companies as a list

In [5]:
# create a company list and quick view first 5 companies
company_list = data['GRNAME'].unique().tolist()
company_list[:5] 

['IDS Property Cas Ins Co',
 'Celina Mut Grp',
 'Federal Ins Co Grp',
 'Buckeye Ins Grp',
 'Employers Mut Co Of Des Moines']

### 3. User Input for a random company

In [6]:
# ask user to type in an integer as company_id to locate a company
# id can be found in Excel sheet
company_id = int(input("Input an integer as company ID"))

Input an integer as company ID 5


In [7]:
# user check if selected company is same as that in Excel
print("Company_id {} is {}".format(company_id, company_list[company_id - 1]))

Company_id 5 is Employers Mut Co Of Des Moines


## Loss Triangle Modelling

In [8]:
# locate company from user's input
company = company_list[company_id-1]

#create a filter with GRNAME=company and DevelopmentYear < 2008
ind = (data["GRNAME"] == company) & (data["DevelopmentYear"] < 2008)
dt = data[ind]

In [9]:
# create loss traingle matrix
loss_triangle = cl.Triangle(data=dt, origin="AccidentYear", development=["DevelopmentYear"], columns=["IncurredLosses"], cumulative=True)
loss_triangle

,12,24,36,48,60,72,84,96,108,120
1998,"63,183","60,017","58,684","58,536","59,547","59,275","59,436","59,330","59,008","58,988"
1999,"65,785","61,262","60,196","60,567","60,649","59,997","59,939","59,953","59,960",
2000,"59,571","54,491","53,932","53,654","53,710","53,894","53,879","53,788",,
2001,"53,875","50,259","50,119","50,798","50,202","50,125","49,962",,,
2002,"52,373","51,455","50,903","50,908","50,500","50,306",,,,
2003,"45,696","47,262","45,548","44,268","44,429",,,,,
2004,"38,756","38,116","38,594","38,017",,,,,,
2005,"37,148","35,113","33,511",,,,,,,
2006,"31,881","30,432",,,,,,,,
2007,"28,721",,,,,,,,,


In [10]:
# heatmap view of loss development factor
loss_triangle.link_ratio.heatmap()

,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
1998,0.9499,0.9778,0.9975,1.0173,0.9954,1.0027,0.9982,0.9946,0.9997
1999,0.9312,0.9826,1.0062,1.0014,0.9892,0.9990,1.0002,1.0001,
2000,0.9147,0.9897,0.9948,1.0010,1.0034,0.9997,0.9983,,
2001,0.9329,0.9972,1.0135,0.9883,0.9985,0.9967,,,
2002,0.9825,0.9893,1.0001,0.9920,0.9962,,,,
2003,1.0343,0.9637,0.9719,1.0036,,,,,
2004,0.9835,1.0125,0.9850,,,,,,
2005,0.9452,0.9544,,,,,,,
2006,0.9545,,,,,,,,


### Chain Ladder Modelling

In [11]:
cl_model = cl.Chainladder().fit(loss_triangle)

# print age-to-ultimate development factor and age-to-age development factor
print(cl_model.cdf_)
print(cl_model.ldf_)

         12-Ult    24-Ult    36-Ult   48-Ult    60-Ult    72-Ult    84-Ult    96-Ult   108-Ult  120-Ult  132-Ult
(All)  0.930256  0.973383  0.989514  0.99292  0.991968  0.995634  0.995968  0.997021  0.999661      1.0      1.0
          12-24     24-36    36-48    48-60     60-72     72-84     84-96    96-108   108-120  120-132  132-144
(All)  0.955694  0.983697  0.99657  1.00096  0.996318  0.999664  0.998944  0.997359  0.999661      1.0      1.0


### Prepare data for final output

In [12]:
data_dict = {"Earned Premium": dt.groupby("AccidentYear")["EarnedPremDIR"].max().values,
             "Incurred Loss as of 2007": loss_triangle.latest_diagonal.values.ravel(), 
             "Cumulative Paid Loss": dt[dt["DevelopmentYear"]==2007]["CumPaidLoss"].values,
             "Age": 2008 - dt[dt["DevelopmentYear"]==2007]["AccidentYear"].values,
             "Age-Ult LDF": cl_model.cdf_.to_frame().iloc[0][::-1][1:].values,
             "Ultimate Loss": cl_model.ultimate_.to_frame().values.ravel(), 
             "IBNR": cl_model.ibnr_.values.ravel()
            }

### Final data output for further reconciliation

In [13]:
output = pd.DataFrame(data_dict, index=loss_triangle.origin)
output

,Earned Premium,Incurred Loss as of 2007,Cumulative Paid Loss,Age,Age-Ult LDF,Ultimate Loss,IBNR
origin,,,,,,,
1998,76045,58988.0,58981,10,1.000000,58988.000000,NaN
1999,78677,59960.0,59900,9,0.999661,59939.677332,-20.322668
2000,74687,53788.0,53584,8,0.997021,53627.775193,-160.224807
2001,70739,49962.0,49037,7,0.995968,49760.556865,-201.443135
2002,71862,50306.0,49316,6,0.995634,50086.341001,-219.658999
2003,69349,44429.0,40929,5,0.991968,44072.146571,-356.853429
2004,59541,38017.0,33320,4,0.992920,37747.853051,-269.146949
2005,51481,33511.0,28449,3,0.989514,33159.611737,-351.388263
2006,45898,30432.0,19636,2,0.973383,29621.980928,-810.019072


In [14]:
# quick view of data by re-indexing
output.reset_index()

,origin,Earned Premium,Incurred Loss as of 2007,Cumulative Paid Loss,Age,Age-Ult LDF,Ultimate Loss,IBNR
0,1998,76045,58988.0,58981,10,1.000000,58988.000000,NaN
1,1999,78677,59960.0,59900,9,0.999661,59939.677332,-20.322668
2,2000,74687,53788.0,53584,8,0.997021,53627.775193,-160.224807
3,2001,70739,49962.0,49037,7,0.995968,49760.556865,-201.443135
4,2002,71862,50306.0,49316,6,0.995634,50086.341001,-219.658999
5,2003,69349,44429.0,40929,5,0.991968,44072.146571,-356.853429
6,2004,59541,38017.0,33320,4,0.992920,37747.853051,-269.146949
7,2005,51481,33511.0,28449,3,0.989514,33159.611737,-351.388263
8,2006,45898,30432.0,19636,2,0.973383,29621.980928,-810.019072
9,2007,44153,28721.0,10811,1,0.930256,26717.879121,-2003.120879


In [35]:
# Write data to Excel file for reconciliation
with pd.ExcelWriter("ppauto_pos98-07_Exercise.xlsm", engine="openpyxl", mode="a", if_sheet_exists="overlay", engine_kwargs={"keep_vba": True}) as writer:
    output.reset_index().to_excel(writer, sheet_name="Summary", startrow=83, startcol=6, index=False)

In [20]:
output.reset_index(names = "AccidentYear").to_excel("Python_Ouput.xlsx", sheet_name="Python", index=False)